In [1]:
import numpy as np
import plotly.graph_objects as go

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader


In [2]:
FAMILY_NAME = "modulo_addition_1layer"
family = load_family(FAMILY_NAME)

REFERENCE_MODELS = [
    (113, 999, 598, "p113 canon"),
    (109, 485, 598, "p109 reference healthy"),
    (101, 999, 598, "p101 open-loop"),
    ( 89, 999, 598, "p89 smooth-diverge"),
    ( 59, 485, 598, "p59 overshooter"),
]

# Qualitative colors — one per frequency group (up to 10 groups), gray for unassigned
_GROUP_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f",
]
_UNASSIGNED_COLOR = "#cccccc"


## Data loading

`neuron_group_pca` stores all MLP neurons projected into the global W_in PCA basis
(fitted on the final epoch) at every checkpoint. Shape: `(n_epochs, d_mlp, 3)`.

Colour = frequency group assignment from the same artifact.

In [3]:
def load_macro_mlp_data(variant) -> dict:
    """Load MLP neuron cloud projected into a single global W_in PCA basis.

    Fits PCA on all d_mlp neurons at the FINAL epoch to get a shared coordinate
    frame, then projects every epoch's W_in into that same basis.  This is a
    true macro view — all neurons share one coordinate system.

    Colour comes from neuron_group_pca (frequency group assignment), but the
    projection itself is independent of group membership.

    Returns:
        projections:  (n_epochs, d_mlp, 3) — all neurons in global W_in PCA space
        epochs:       (n_epochs,)
        group_idx:    (d_mlp,) — group index per neuron; -1 = ungrouped
        group_freqs:  (n_groups,) — 1-indexed frequency per group
        var_ratio:    (3,) — explained variance fraction of the global PCA
    """
    from sklearn.decomposition import PCA
    from miscope.analysis.library import extract_neuron_weight_matrix

    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))

    # Group colouring from neuron_group_pca (unchanged)
    ngpca = loader.load_cross_epoch("neuron_group_pca")
    group_idx   = ngpca["neuron_group_idx"]          # (d_mlp,)
    group_freqs = (ngpca["group_freqs"] + 1).astype(int)  # 1-indexed
    epochs_arr  = ngpca["epochs"]                    # (n_epochs,)

    # Global PCA: fit on final epoch, project all epochs
    epoch_list = sorted(epochs_arr.tolist())
    snaps = [loader.load_epoch("parameter_snapshot", e) for e in epoch_list]

    final_W = extract_neuron_weight_matrix(snaps[-1])  # (d_model, d_mlp)
    pca = PCA(n_components=3, random_state=0)
    pca.fit(final_W.T)   # fit on (d_mlp, d_model) — each neuron is a sample

    projections = np.stack(
        [pca.transform(extract_neuron_weight_matrix(s).T) for s in snaps]
    ).astype(np.float32)  # (n_epochs, d_mlp, 3)

    return {
        "projections": projections,
        "epochs":      epochs_arr,
        "group_idx":   group_idx,
        "group_freqs": group_freqs,
        "var_ratio":   pca.explained_variance_ratio_,
    }


## Animation builder

One 3-D scatter per frequency group, animated across epochs.

Axis ranges are fixed to the global extent (all epochs, all neurons) so
the camera stays still while the cloud evolves.

Initial view = final epoch (most organised). Slider scrubs backward.

In [4]:
def make_macro_surface_animation(variant, label: str) -> go.Figure:
    d = load_macro_mlp_data(variant)
    proj      = d["projections"]   # (n_epochs, d_mlp, 3)
    epochs    = d["epochs"]
    group_idx = d["group_idx"]
    group_freqs = d["group_freqs"]

    n_epochs, d_mlp, _ = proj.shape
    n_groups = len(group_freqs)

    # Fixed axis range across all epochs
    flat = proj.reshape(-1, 3)
    pad = 0.05
    ranges = []
    for dim in range(3):
        lo, hi = flat[:, dim].min(), flat[:, dim].max()
        margin = (hi - lo) * pad
        ranges.append([float(lo - margin), float(hi + margin)])

    # Group masks (computed once)
    masks = [group_idx == g for g in range(n_groups)]
    mask_unassigned = group_idx == -1

    # ── Initial traces (final epoch) ──────────────────────────────────────────
    init_ep = n_epochs - 1
    traces = []
    for g in range(n_groups):
        pts = proj[init_ep][masks[g]]
        color = _GROUP_COLORS[g % len(_GROUP_COLORS)]
        traces.append(go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=dict(size=3, color=color, opacity=0.75),
            name=f"freq {group_freqs[g]}",
        ))
    if mask_unassigned.any():
        pts = proj[init_ep][mask_unassigned]
        traces.append(go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=dict(size=2, color=_UNASSIGNED_COLOR, opacity=0.4),
            name="unassigned",
        ))

    n_traces = len(traces)

    # ── Animation frames ──────────────────────────────────────────────────────
    frames = []
    for ep_i in range(n_epochs):
        frame_data = []
        for g in range(n_groups):
            pts = proj[ep_i][masks[g]]
            frame_data.append(dict(
                type="scatter3d",
                x=pts[:, 0].tolist(),
                y=pts[:, 1].tolist(),
                z=pts[:, 2].tolist(),
            ))
        if mask_unassigned.any():
            pts = proj[ep_i][mask_unassigned]
            frame_data.append(dict(
                type="scatter3d",
                x=pts[:, 0].tolist(),
                y=pts[:, 1].tolist(),
                z=pts[:, 2].tolist(),
            ))
        frames.append(go.Frame(
            data=frame_data,
            name=str(int(epochs[ep_i])),
        ))

    # ── Slider ────────────────────────────────────────────────────────────────
    slider_steps = [
        {
            "args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
            "method": "animate",
            "label": f.name,
        }
        for f in frames
    ]
    sliders = [{
        "steps":        slider_steps,
        "active":       n_epochs - 1,   # start at final epoch
        "currentvalue": {"prefix": "Epoch: ", "visible": True, "xanchor": "left"},
        "len": 0.88, "x": 0.06, "y": 0.0,
        "pad": {"t": 50},
    }]

    updatemenus = [{
        "type": "buttons",
        "showactive": False,
        "y": 0.02, "x": 0.0, "xanchor": "left",
        "buttons": [
            {"label": "▶ Play",
             "method": "animate",
             "args": [None, {"frame": {"duration": 120, "redraw": True},
                             "fromcurrent": True, "transition": {"duration": 0}}]},
            {"label": "⏸ Pause",
             "method": "animate",
             "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]},
        ],
    }]

    fig = go.Figure(data=traces, frames=frames)
    fig.update_layout(
        title=f"{label} — MLP neuron cloud (global W_in PCA, var: {d['var_ratio']*100})",
        scene=dict(
            xaxis=dict(title="PC1", range=ranges[0]),
            yaxis=dict(title="PC2", range=ranges[1]),
            zaxis=dict(title="PC3", range=ranges[2]),
            aspectmode="cube",
        ),
        sliders=sliders,
        updatemenus=updatemenus,
        height=680, width=820,
        margin=dict(l=0, r=0, b=100, t=50),
        legend=dict(x=1.0, y=0.9),
    )
    return fig


In [6]:
v = family.get_variant(prime=113, seed=999, data_seed=598)
make_macro_surface_animation(v, "p113 canon").show()


In [ ]:
v = family.get_variant(prime=109, seed=485, data_seed=598)
make_macro_surface_animation(v, "p109 reference healthy").show()


In [ ]:
v = family.get_variant(prime=101, seed=999, data_seed=598)
make_macro_surface_animation(v, "p101 open-loop").show()


In [ ]:
v = family.get_variant(prime=89, seed=999, data_seed=598)
make_macro_surface_animation(v, "p89 smooth-diverge").show()


In [ ]:
v = family.get_variant(prime=59, seed=485, data_seed=598)
make_macro_surface_animation(v, "p59 overshooter").show()
